In [ ]:
from futu import *
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)
ret, data = quote_ctx.get_global_state()
if ret == RET_OK:
    print(data) # 检查其中的 'qutotation_permission' 字段
quote_ctx.close()

In [4]:
from futu import *

# 1. 建立连接
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)

# 2. 设定 PDD 的 DCF 提醒逻辑
# 假设 dcf_target_price 是你计算出来的买入价
dcf_target_price = 66.5 

ret, data = quote_ctx.set_price_reminder(
    code='US.PDD', 
    op=SetPriceReminderOp.ADD, 
    reminder_type=PriceReminderType.PRICE_DOWN, # 价格跌到
    value=dcf_target_price, 
    note='DCF 24x 估值点',
    # 修正这里：设置为每日一次
    reminder_freq=PriceReminderFreq.ONCE_A_DAY 
)

if ret == RET_OK:
    # 注意：这里的 data 直接就是 ID 整数
    print(f"PDD 提醒设置成功！每日触发一次。ID: {data}")
else:
    # 如果失败，这里的 data 通常是错误字符串
    print(f"设置失败，原因: {data}")

# 3. 关闭连接
quote_ctx.close()

2026-04-10 23:39:49,718 | 32212 | 33040 | [open_context_base.py:409] _init_connect_sync: New connect ready: conn=7448620809716234093(4) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000296D5734970>
PDD 提醒设置成功！每日触发一次。ID: 1775889589572863002
2026-04-10 23:39:49,818 | 32212 | 28744 | [open_context_base.py:516] on_disconnect: Disconnected: conn=0(4) reason=CallClose


In [3]:
import akshare as ak
import pandas as pd

def get_fcf_smart(symbol="600519"):
    print(f"正在分析 {symbol} 的财务数据...")
    full_symbol = f"sh{symbol}" if symbol.startswith("60") else f"sz{symbol}"
    
    # 获取数据
    df = ak.stock_financial_report_sina(stock=full_symbol, symbol="现金流量表")
    
    # 打印前 5 个列名，方便排错
    columns = df.columns.tolist()
    print(f"【Debug】成功获取数据，表头包含: {columns[:5]}...")

    # 1. 智能寻找日期列
    date_col = next((c for c in columns if "期" in c or "日" in c), None)
    
    # 2. 智能寻找经营现金流列
    ocf_col = next((c for c in columns if "经营活动" in c and "净额" in c), None)
    
    # 3. 智能寻找资本支出列
    capex_col = next((c for c in columns if "固定资产" in c and "支付" in c), None)

    if not all([date_col, ocf_col, capex_col]):
        print("【错误】未能匹配到所需科目，请检查上方打印的表头！")
        return None
        
    print(f"【匹配成功】日期列: {date_col} | OCF列: {ocf_col} | CapEx列: {capex_col}")

    # 数据清洗
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df_annual = df[df[date_col].dt.month == 12].copy()
    
    df_annual[ocf_col] = pd.to_numeric(df_annual[ocf_col], errors='coerce')
    df_annual[capex_col] = pd.to_numeric(df_annual[capex_col], errors='coerce')
    
    # 计算 FCF
    df_annual["FCF_亿元"] = (df_annual[ocf_col] - df_annual[capex_col]) / 1e8
    df_annual["OCF_亿元"] = df_annual[ocf_col] / 1e8
    df_annual["CapEx_亿元"] = df_annual[capex_col] / 1e8
    
    return df_annual[[date_col, "OCF_亿元", "CapEx_亿元", "FCF_亿元"]].head(10)

if __name__ == "__main__":
    data = get_fcf_smart("600519")
    if data is not None:
        print("\n自由现金流分析结果 (最近10年)：")
        # 重命名日期列使其更易读
        data.rename(columns={data.columns[0]: "年份"}, inplace=True)
        # 将日期格式化为 YYYY-MM-DD
        data["年份"] = data["年份"].dt.strftime('%Y-%m-%d')
        print(data.to_string(index=False))

正在分析 600519 的财务数据...
【Debug】成功获取数据，表头包含: ['报告日', '经营活动产生的现金流量', '销售商品、提供劳务收到的现金', '客户存款和同业存放款项净增加额', '向中央银行借款净增加额']...
【匹配成功】日期列: 报告日 | OCF列: 经营活动产生的现金流量净额 | CapEx列: 购建固定资产、无形资产和其他长期资产所支付的现金

自由现金流分析结果 (最近10年)：
        年份     OCF_亿元  CapEx_亿元     FCF_亿元
2024-12-31 924.636922 46.787121 877.849801
2023-12-31 665.932477 26.197559 639.734918
2022-12-31 366.985958 53.065464 313.920494
2021-12-31 640.286761 34.087845 606.198916
2020-12-31 516.690687 20.897695 495.792992
2019-12-31 452.106126 31.488647 420.617480
2018-12-31 413.852344 16.067502 397.784842
2017-12-31 221.530361 11.250172 210.280189
2016-12-31 374.512496 10.191781 364.320715
2015-12-31 174.363401 20.614705 153.748697


# A股数据查询示例 — 茅台 600519

分两部分：**akshare**（免费，无需注册）和 **tushare**（需要 token）。

## Part 1 — akshare（免费）

In [ ]:
import akshare as ak
import pandas as pd

SYMBOL = "600519"  # 贵州茅台

# ── 1. 实时行情快照 ─────────────────────────────────────────────
# 返回：代码、名称、最新价、涨跌幅、成交量、市值等
spot = ak.stock_zh_a_spot_em()
row = spot[spot["代码"] == SYMBOL]
print("=== 实时行情 ===")
print(row[["代码","名称","最新价","涨跌幅","总市值","市盈率-动态"]].to_string(index=False))

In [ ]:
# ── 2. 日K线（前复权）─────────────────────────────────────────
# period: daily / weekly / monthly
# adjust: "" 不复权 | "qfq" 前复权 | "hfq" 后复权
kline = ak.stock_zh_a_hist(
    symbol=SYMBOL,
    period="daily",
    start_date="20200101",
    end_date="20241231",
    adjust="qfq",
)
print("=== 日K线（前复权，最近5条）===")
print(kline.tail())

In [ ]:
# ── 3. 利润表（新浪财经）──────────────────────────────────────
# symbol 可选: "利润表" | "资产负债表" | "现金流量表"
profit = ak.stock_financial_report_sina(stock=f"sh{SYMBOL}", symbol="利润表")
print("=== 利润表列名 ===")
print(profit.columns.tolist()[:10])
print()
# 取关键指标
cols_want = ["报告日","营业总收入","营业利润","净利润"]
available = [c for c in cols_want if c in profit.columns]
print(profit[available].head(6).to_string(index=False))

In [ ]:
# ── 4. 资产负债表 ─────────────────────────────────────────────
balance = ak.stock_financial_report_sina(stock=f"sh{SYMBOL}", symbol="资产负债表")
cols_want = ["报告日","货币资金","应收账款","总资产","总负债","股东权益合计"]
available = [c for c in cols_want if c in balance.columns]
print("=== 资产负债表（精简）===")
print(balance[available].head(6).to_string(index=False))

In [ ]:
# ── 5. 股票基本信息（东方财富）────────────────────────────────
info = ak.stock_individual_info_em(symbol=SYMBOL)
# 返回 DataFrame with columns: item, value
print("=== 个股基本信息 ===")
print(info.to_string(index=False))

In [ ]:
# ── 6. 分红派息历史 ───────────────────────────────────────────
dividend = ak.stock_divident_cninfo(symbol=SYMBOL)
print("=== 历史分红（最近5次）===")
print(dividend.head())

In [ ]:
# ── 7. 机构持股（龙虎榜 / 主力资金）─────────────────────────
# 每日主力净流入（东方财富）
flow = ak.stock_individual_fund_flow(stock=SYMBOL, market="sh")
print("=== 主力资金流向（最近5天）===")
print(flow.tail())

## Part 2 — tushare（需要 token）

先在 `.env` 里设置 `TUSHARE_TOKEN=xxx`，或直接 `ts.set_token("xxx")`。

In [ ]:
import tushare as ts
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env 文件
ts.set_token(os.getenv("TUSHARE_TOKEN", ""))
pro = ts.pro_api()

TS_CODE = "600519.SH"  # tushare 格式：代码.交易所

In [ ]:
# ── 1. 股票基本信息 ───────────────────────────────────────────
basic = pro.stock_basic(ts_code=TS_CODE,
                        fields="ts_code,name,area,industry,list_date,market")
print("=== 基本信息 ===")
print(basic.to_string(index=False))

In [ ]:
# ── 2. 日K线（后复权）─────────────────────────────────────────
# adj: None 不复权 | "qfq" 前复权 | "hfq" 后复权
daily = pro.daily(ts_code=TS_CODE, start_date="20240101", end_date="20241231")
print("=== 日K线（最近5条）===")
print(daily.head())

In [ ]:
# ── 3. 复权因子（用来自己算复权价）─────────────────────────
adj_factor = pro.adj_factor(ts_code=TS_CODE, start_date="20240101")
print("=== 复权因子（最近5条）===")
print(adj_factor.head())

In [ ]:
# ── 4. 每日基本面指标（PE/PB/市值等）────────────────────────
daily_basic = pro.daily_basic(
    ts_code=TS_CODE,
    start_date="20240101",
    end_date="20241231",
    fields="trade_date,pe,pe_ttm,pb,ps_ttm,total_mv,circ_mv,turnover_rate",
)
print("=== 每日基本面（最近5条）===")
print(daily_basic.head())

In [ ]:
# ── 5. 利润表（年报）─────────────────────────────────────────
# report_type: 1=合并报表 | period: 年报末尾是 1231
income = pro.income(
    ts_code=TS_CODE,
    period="20231231",
    report_type="1",
    fields="ts_code,end_date,total_revenue,operate_profit,n_income_attr_p",
)
print("=== 年度利润表 ===")
print(income.to_string(index=False))

In [ ]:
# ── 6. 资产负债表 ─────────────────────────────────────────────
balance = pro.balancesheet(
    ts_code=TS_CODE,
    period="20231231",
    report_type="1",
    fields="ts_code,end_date,money_cap,total_assets,total_liab,total_hldr_eqy_exc_min_int",
)
print("=== 资产负债表 ===")
print(balance.to_string(index=False))

In [ ]:
# ── 7. 现金流量表（算 FCF）─────────────────────────────────
cashflow = pro.cashflow(
    ts_code=TS_CODE,
    period="20231231",
    report_type="1",
    fields="ts_code,end_date,n_cashflow_act,c_pay_acq_const_fiolta",
)
# n_cashflow_act       = 经营活动净现金流
# c_pay_acq_const_fiolta = 购建固定资产等支付的现金（CapEx）
cashflow["FCF"] = cashflow["n_cashflow_act"] - cashflow["c_pay_acq_const_fiolta"]
print("=== 现金流量表 + FCF ===")
print(cashflow.to_string(index=False))

In [ ]:
# ── 8. 分红送股记录 ───────────────────────────────────────────
dividend = pro.dividend(ts_code=TS_CODE, fields="ts_code,end_date,cash_div_tax,stk_div")
print("=== 历史分红（最近5条）===")
print(dividend.head())

In [ ]:
# ── 9. 主要财务指标（ROE / EPS / 每股FCF 等）──────────────
fina = pro.fina_indicator(
    ts_code=TS_CODE,
    period="20231231",
    fields="ts_code,end_date,roe,eps,bps,netprofit_margin,grossprofit_margin,fcff",
)
# fcff = 企业自由现金流量（FCFF），单位：元/股（部分版本可能是总额，视 token 权限而定）
print("=== 主要财务指标 ===")
print(fina.to_string(index=False))